# 03 — Sequence models on FD001 (LSTM and TransLSTM)

In `02_baselines` every model saw **one cycle at a time**. Single sensor readings are noisy, and one cycle does not show the *trend* of the degradation.

Here we give the model **the last 30 cycles** of an engine (a *window*) and let a neural network learn from the whole sequence. We train two PyTorch models:

1. a simple **LSTM**;
2. the **TransLSTM** (Transformer encoder + LSTM), based on the course example.

We then compare them on the test set with the best baseline from `02` (**Random Forest: RMSE 17.58, NASA score 843**).

Steps:
1. Same preprocessing as `02`: 14 sensors, RUL capped at 125, train/validation split by engine.
2. Scale the sensors (scaler fitted on the training engines only).
3. Build sliding windows of 30 cycles.
4. Train both models with early stopping and plot the loss curves.
5. Evaluate on the test set and compare with the baseline.

The reusable code is in `src/`: `data.py` (loading, split), `sequences.py` (windows), `models.py` (the two networks), `training.py` (training loop) and `metrics.py`.

## 0. Setup

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.ticker import MaxNLocator
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# Project root = parent of the "notebooks" folder
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data import FD001_SENSORS, add_train_rul, load_cmapss, split_by_engine
from src.metrics import nasa_score, rmse
from src.models import LSTMModel, TransLSTM, count_parameters
from src.sequences import make_last_windows, make_windows
from src.training import predict, set_seed, train_model

FIG_DIR = PROJECT_ROOT / "results" / "figures"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
MODELS_DIR = PROJECT_ROOT / "results" / "models"
for folder in [FIG_DIR, METRICS_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
RUL_CAP = 125
WINDOW_SIZE = 30

set_seed(SEED)
print("PyTorch", torch.__version__, "| CPU threads:", torch.get_num_threads())

## 1. Load the data, compute and cap the RUL

Exactly as in `02_baselines`:
- load FD001 and compute the RUL of every training row;
- cap the RUL at 125 cycles (`min(RUL, 125)`), because the sensors cannot tell how healthy a healthy engine is;
- keep the 14 informative sensors from `01_eda` (`FD001_SENSORS`).

On the test set we still compare against the **true, uncapped** RUL from the RUL file, as in `02`.

In [ ]:
train, test, test_rul = load_cmapss("FD001")
train = add_train_rul(train)
train["RUL"] = np.minimum(train["RUL"], RUL_CAP)

y_test = test_rul["true_rul"].values
print("Train rows:", len(train), "| Test engines:", test["unit"].nunique())

## 2. Split train / validation by engine

`split_by_engine` (in `src/data.py`) puts 80 whole engines in training and 20 in validation, with the same seed as `02`, so it gives **the same engines** as the baseline. The validation set is used for **early stopping** only; the final comparison is on the official test set.

In [ ]:
train_part, val_part = split_by_engine(train, val_fraction=0.2, seed=SEED)

print("Train engines:", train_part["unit"].nunique(), "| Validation engines:", val_part["unit"].nunique())
print("Engines in both sets:", len(set(train_part["unit"]) & set(val_part["unit"])))

## 3. Scale the sensors

Neural networks train badly when inputs have very different scales (sensor_9 ≈ 9000, sensor_15 ≈ 8.4). We use a `StandardScaler`: for every sensor, subtract its mean and divide by its standard deviation.

The scaler is **fitted on the training engines only**, then applied to validation and test. If we fitted it on all the data, information about the validation/test engines would leak into training.

In [ ]:
scaler = StandardScaler()
scaler.fit(train_part[FD001_SENSORS])  # learn mean and std on training engines only

# .copy() so that we do not modify the original data frames
train_scaled = train_part.copy()
val_scaled = val_part.copy()
test_scaled = test.copy()
train_scaled[FD001_SENSORS] = scaler.transform(train_part[FD001_SENSORS])
val_scaled[FD001_SENSORS] = scaler.transform(val_part[FD001_SENSORS])
test_scaled[FD001_SENSORS] = scaler.transform(test[FD001_SENSORS])

train_scaled[FD001_SENSORS].describe().loc[["mean", "std"]].round(2)

## 4. Sliding windows of 30 cycles

Each sample is now a **window** of 30 consecutive cycles of one engine (shape 30 × 14). The target is the (capped) RUL at the **last** cycle of the window: the model looks at the recent history and predicts the RUL *now*.

For an engine with `n` cycles we get `n − 29` overlapping windows: cycles 1–30, 2–31, …, (n−29)–n. Windows never mix two engines.

- **Train / validation:** all windows of every engine (`make_windows`).
- **Test:** only the **last** window of each test engine (`make_last_windows`), because the true RUL is known only at the last recorded cycle. Every FD001 test engine has at least 31 cycles, so a full 30-cycle window always exists.

**Target scaling:** we divide the RUL by 125, so the network predicts a number between 0 and 1 (easier to train). We multiply the predictions by 125 to get back to cycles.

In [ ]:
X_train, y_train = make_windows(train_scaled, FD001_SENSORS, WINDOW_SIZE)
X_val, y_val = make_windows(val_scaled, FD001_SENSORS, WINDOW_SIZE)
X_test, test_units = make_last_windows(test_scaled, FD001_SENSORS, WINDOW_SIZE)

# Check that the test windows are in the same engine order as the RUL file
assert (test_units == test_rul["unit"].values).all()

# Scale the targets to [0, 1]
y_train_scaled = y_train / RUL_CAP
y_val_scaled = y_val / RUL_CAP

print("X_train:", X_train.shape, " (windows, cycles, sensors)")
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

## 5. Model 1: LSTM

An **LSTM** (Long Short-Term Memory) reads the window one cycle at a time and keeps an internal memory (hidden state) that summarises what it has seen so far. After the 30th cycle, the hidden state is passed to a linear layer that outputs the RUL.

Our LSTM: 2 stacked LSTM layers with 64 hidden units, dropout 0.2 between them, then one linear output layer.

**Training** (`train_model` in `src/training.py`):
- loss = mean squared error (MSE), optimizer = Adam (learning rate 0.001), mini-batches of 256 windows;
- **early stopping:** after every epoch we compute the validation loss; if it has not improved for 10 epochs, we stop and restore the weights of the best epoch (maximum 60 epochs);
- all random seeds are fixed, so the result is reproducible.

In [ ]:
set_seed(SEED)  # same initial weights every run
lstm = LSTMModel(n_features=len(FD001_SENSORS), hidden_size=64, num_layers=2, dropout=0.2)
print("LSTM trainable parameters:", count_parameters(lstm))

start = time.time()
lstm, lstm_history = train_model(lstm, X_train, y_train_scaled, X_val, y_val_scaled, seed=SEED)
print(f"Training time: {time.time() - start:.0f} s")

## 6. Model 2: TransLSTM (Transformer encoder + LSTM)

The TransLSTM combines two ideas:

1. **Input projection:** a linear layer maps the 14 sensors of each cycle to a vector of 64 numbers (`d_model`).
2. **Positional embedding:** a learned vector per position (1…30) is added, so the model knows the order of the cycles. Self-attention on its own ignores order.
3. **Transformer encoder** (2 layers, 4 attention heads): with **self-attention**, every cycle in the window can directly "look at" every other cycle and decide which ones are important, instead of passing information step by step.
4. **LSTM** (64 units) reads the encoded sequence in time order; its last hidden state goes to a linear layer that outputs the RUL.

It is trained with exactly the same procedure and settings as the LSTM.

> **Note:** the course's TransLSTM example is not in this repository. This implementation follows the architecture described above (Transformer encoder, then LSTM, then a linear output), adapted to our input of 30 cycles × 14 sensors. Layer sizes may differ from the course example.

In [ ]:
set_seed(SEED)
translstm = TransLSTM(
    n_features=len(FD001_SENSORS),
    window_size=WINDOW_SIZE,
    d_model=64,
    n_heads=4,
    n_encoder_layers=2,
    lstm_hidden_size=64,
    dropout=0.1,
)
print("TransLSTM trainable parameters:", count_parameters(translstm))

start = time.time()
translstm, translstm_history = train_model(translstm, X_train, y_train_scaled, X_val, y_val_scaled, seed=SEED)
print(f"Training time: {time.time() - start:.0f} s")

## 7. Loss curves

To make the curves easier to read, we convert the MSE loss (on the 0–1 scale) back to **RMSE in cycles**: `RMSE = sqrt(MSE) × 125`. The dashed line marks the best validation epoch (the weights we keep).

Note: the training loss is computed *with* dropout switched on, so it can be slightly higher than the validation loss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

for ax, history, name in [
    (axes[0], lstm_history, "LSTM"),
    (axes[1], translstm_history, "TransLSTM"),
]:
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    train_rmse = np.sqrt(history["train_loss"]) * RUL_CAP
    val_rmse = np.sqrt(history["val_loss"]) * RUL_CAP
    best_epoch = int(np.argmin(history["val_loss"])) + 1

    ax.plot(epochs, train_rmse, label="Train")
    ax.plot(epochs, val_rmse, label="Validation")
    ax.axvline(best_epoch, color="grey", linestyle="--", label=f"Best epoch ({best_epoch})")
    ax.set_xlabel("Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))  # whole epochs only
    ax.set_title(name)
    ax.legend()

axes[0].set_ylabel("RMSE (cycles)")
axes[0].set_ylim(0, 40)
plt.tight_layout()
plt.savefig(FIG_DIR / "03_loss_curves.png", dpi=150)
plt.show()

## 8. Evaluation on the test set

We predict the RUL of the 100 test engines from their last window, multiply by 125 to get cycles, and compute RMSE and the NASA score against the true RUL.

We also report the **validation RMSE** (in cycles, against the capped target) to see if validation and test agree.

For the comparison, we **retrain the Random Forest of `02`** (Step 2: engine split + cap 125, one cycle per sample, same settings) to have its individual predictions for the plot. It should reproduce the RMSE of 17.58.

In [ ]:
# Sequence models: predictions back to cycles
lstm_test_pred = predict(lstm, X_test) * RUL_CAP
translstm_test_pred = predict(translstm, X_test) * RUL_CAP
lstm_val_pred = predict(lstm, X_val) * RUL_CAP
translstm_val_pred = predict(translstm, X_val) * RUL_CAP

# Random Forest baseline, exactly as in 02 (Step 2): raw sensors of the last cycle
rf = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
rf.fit(train_part[FD001_SENSORS], train_part["RUL"])
test_last_cycle = test.groupby("unit").tail(1).sort_values("unit")
rf_test_pred = rf.predict(test_last_cycle[FD001_SENSORS])
rf_val_pred = rf.predict(val_part[FD001_SENSORS])

In [ ]:
predictions = {
    "Random Forest (02, best baseline)": (rf_val_pred, val_part["RUL"], rf_test_pred),
    "LSTM": (lstm_val_pred, y_val, lstm_test_pred),
    "TransLSTM": (translstm_val_pred, y_val, translstm_test_pred),
}

rows = []
for name, (val_pred, val_true, test_pred) in predictions.items():
    rows.append({
        "model": name,
        "val_RMSE": rmse(val_true, val_pred),
        "test_RMSE": rmse(y_test, test_pred),
        "test_NASA_score": nasa_score(y_test, test_pred),
    })

comparison = pd.DataFrame(rows)
comparison.to_csv(METRICS_DIR / "03_sequence_models_fd001.csv", index=False)
comparison.round(2)

*Note:* the validation RMSE of the Random Forest uses **every cycle** of the validation engines, while the sequence models only have windows from cycle 30 onwards, so the validation columns are not exactly comparable. The **test** columns are: all three models predict the same 100 engines.

### Predicted vs true RUL

Points on the dashed diagonal are perfect predictions. Points **above** it are **late** predictions (the model thinks the engine has more life than it really has: the dangerous case); points below are early predictions. The dotted line marks the cap of 125: no model can predict much above it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)

for ax, (name, (_, _, test_pred)) in zip(axes, predictions.items()):
    ax.scatter(y_test, test_pred, alpha=0.7, edgecolor="black")
    ax.plot([0, 150], [0, 150], "k--", linewidth=1)          # perfect prediction
    ax.axhline(RUL_CAP, color="grey", linestyle=":", linewidth=1)
    test_rmse = rmse(y_test, test_pred)
    ax.set_title(f"{name}\nRMSE {test_rmse:.2f}")
    ax.set_xlabel("True RUL")

axes[0].set_ylabel("Predicted RUL")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_pred_vs_true.png", dpi=150)
plt.show()

## 9. Save the trained models

We save the weights (and the scaler) so that later notebooks (SHAP explainability, maintenance decision rule) can reuse the models without retraining.

In [ ]:
import joblib

torch.save(lstm.state_dict(), MODELS_DIR / "03_lstm_fd001.pt")
torch.save(translstm.state_dict(), MODELS_DIR / "03_translstm_fd001.pt")
joblib.dump(scaler, MODELS_DIR / "03_scaler_fd001.joblib")
print("Saved to", MODELS_DIR)

## 10. How much do the results depend on the random seed?

A neural network's result depends on its random initial weights and on the random order of the mini-batches. With only 100 test engines, a difference of less than a cycle between two models could just be luck. To check this, we retrain both models with **3 other seeds** (0, 1, 2), with everything else unchanged.

This takes about 8 minutes on a laptop CPU, so it is **switched off by default**. Set `RUN_SEED_CHECK = True` to run it. The results we obtained are in the table below the cell.

In [ ]:
RUN_SEED_CHECK = False

if RUN_SEED_CHECK:
    seed_rows = []
    for seed in [0, 1, 2]:
        for name in ["LSTM", "TransLSTM"]:
            set_seed(seed)
            if name == "LSTM":
                model = LSTMModel(n_features=len(FD001_SENSORS))
            else:
                model = TransLSTM(n_features=len(FD001_SENSORS), window_size=WINDOW_SIZE)
            model, _ = train_model(model, X_train, y_train_scaled, X_val, y_val_scaled, seed=seed)
            test_pred = predict(model, X_test) * RUL_CAP
            seed_rows.append({
                "seed": seed,
                "model": name,
                "test_RMSE": rmse(y_test, test_pred),
                "test_NASA_score": nasa_score(y_test, test_pred),
            })
    display(pd.DataFrame(seed_rows).round(2))

Results we obtained (seed 42 = the run above):

| Seed | LSTM RMSE | LSTM NASA | TransLSTM RMSE | TransLSTM NASA |
|---|---|---|---|---|
| 0 | 15.14 | 372 | 15.16 | 340 |
| 1 | 15.90 | 387 | 15.15 | 531 |
| 2 | 15.79 | 477 | 14.61 | 335 |
| 42 | 15.93 | 465 | 15.29 | 364 |
| **Range** | **15.1 – 15.9** | **372 – 477** | **14.6 – 15.3** | **335 – 531** |

With every seed, both sequence models beat the Random Forest (17.58 / 843).

## Discussion

1. **Time windows clearly help.** With every seed, both sequence models beat the Random Forest: test RMSE goes from 17.6 to about 15, and the NASA score from 843 to between 335 and 531.

2. **TransLSTM vs LSTM.** The TransLSTM has a slightly lower RMSE in 3 of the 4 seeds, but the NASA score ranges overlap. Its advantage is small and not conclusive.

3. **Loss curves.** For the LSTM, train and validation errors go down together until about epoch 8; after that the validation error levels off. The TransLSTM overfits after epoch 3: its train error keeps falling while the validation error rises. Early stopping keeps the epoch-3 model, so this overfitting does not reach the final model.

4. **Validation RMSE (~12) is lower than test RMSE (~15)** for two main reasons:
   - Validation uses *all* windows of each engine, and about 30% of them are in the easy capped plateau (target = 125). Test only uses the *last* window of each engine.
   - We chose the best epoch on the validation set, so its score is a little optimistic.

   The cap explains only part of the gap: even counting only test engines with a true RUL of 125 or less, the LSTM's test RMSE is 15.3.